# Pipeline ISBN CollectionLens

## Objectif

Valider le premier pipeline de récupération multi-sources.

Le pipeline doit :

1. récupérer les métadonnées depuis plusieurs sources ;
2. centraliser les résultats ;
3. préparer la future agrégation multi-sources ;
4. préparer la future couche de cache local ;
5. préparer la sauvegarde des données brutes.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT / "src"))

In [2]:
from collectionlens.pipelines.isbn_pipeline import fetch_isbn_metadata

from collectionlens.api.google_books_api import (
    search_book_by_isbn as google_books_isbn,
)

from collectionlens.api.bnf_api import (
    search_book_by_isbn as bnf_isbn,
)

from collectionlens.api.open_library_api import (
    search_book_by_isbn as openlibrary_isbn,
)

from collectionlens.api.nudger_dataset import (
    load_nudger_dataset,
    build_nudger_search_function,
)

In [3]:
nudger_path = (
    PROJECT_ROOT
    / "data"
    / "external"
    / "nudger"
    / "open4goods-isbn-dataset.csv"
)

df_nudger_source = load_nudger_dataset(
    csv_path=nudger_path,
    isbn_column="isbn",
)

nudger_isbn = build_nudger_search_function(df_nudger_source)

In [4]:
sources = {
    "nudger": nudger_isbn,
    "google_books": google_books_isbn,
    "bnf": bnf_isbn,
    "openlibrary": openlibrary_isbn,
}

In [5]:
result = fetch_isbn_metadata(
    isbn="9782351423554",
    sources=sources,
)

result.keys()

dict_keys(['isbn_query', 'retrieved_at', 'sources'])

In [6]:
result["sources"].keys()

dict_keys(['nudger', 'google_books', 'bnf', 'openlibrary'])

In [7]:
for source_name, source_result in result["sources"].items():
    print(source_name, ":", source_result.get("found"), "-", source_result.get("title"))

nudger : True - vinland saga - tome
google_books : False - None
bnf : True - Vinland saga. 1 / Makoto Yukimura
openlibrary : True - Vinland Saga (1)


In [8]:
test_isbns = [
    "9782351423554",
    "9782203001190",
    "9791039143431",
]

In [9]:
from collectionlens.pipelines.isbn_pipeline import (
    fetch_many_isbns_metadata,
)

In [10]:
results = fetch_many_isbns_metadata(
    isbns=test_isbns,
    sources=sources,
)

In [11]:
len(results)

3

In [12]:
for result in results:

    print("\n", result["isbn_query"])

    for source_name, source_result in result["sources"].items():

        print(
            f"{source_name:<15}"
            f"{source_result.get('found')}"
        )


 9782351423554
nudger         True
google_books   False
bnf            True
openlibrary    True

 9782203001190
nudger         True
google_books   False
bnf            False
openlibrary    True

 9791039143431
nudger         True
google_books   False
bnf            True
openlibrary    False


In [13]:
for result in results:

    print("\nISBN :", result["isbn_query"])

    google_result = result["sources"]["google_books"]

    print("found :", google_result.get("found"))
    print("error :", google_result.get("error"))
    print("status :", google_result.get("status_code"))


ISBN : 9782351423554
found : False
error : quota_exceeded
status : 429

ISBN : 9782203001190
found : False
error : quota_exceeded
status : 429

ISBN : 9791039143431
found : False
error : quota_exceeded
status : 429


Le test multi-ISBN confirme que le pipeline centralise correctement les résultats et les erreurs par source.

Google Books retourne ici une erreur `quota_exceeded` avec un statut HTTP 429. Ce comportement confirme l’intérêt de la future stratégie de cache local ISBN afin d’éviter les appels répétés aux APIs externes et de limiter l’impact des quotas.

Le pipeline est donc validé sur sa capacité à :
- interroger plusieurs sources ;
- retourner les résultats par source ;
- conserver les erreurs API ;
- préparer l’intégration du cache local.

In [14]:
raw_output_dir = PROJECT_ROOT / "data" / "raw" / "isbn_pipeline"

result = fetch_isbn_metadata(
    isbn="9782351423554",
    sources=sources,
    raw_output_dir=raw_output_dir,
)

In [15]:
results = fetch_many_isbns_metadata(
    isbns=test_isbns,
    sources=sources,
    raw_output_dir=raw_output_dir,
)

In [16]:
list(raw_output_dir.rglob("*.json"))[:10]

[WindowsPath('c:/Users/yoann/Documents/mes projets/CollectionLens V2/data/raw/isbn_pipeline/bnf/9782203001190.json'),
 WindowsPath('c:/Users/yoann/Documents/mes projets/CollectionLens V2/data/raw/isbn_pipeline/bnf/9782351423554.json'),
 WindowsPath('c:/Users/yoann/Documents/mes projets/CollectionLens V2/data/raw/isbn_pipeline/bnf/9791039143431.json'),
 WindowsPath('c:/Users/yoann/Documents/mes projets/CollectionLens V2/data/raw/isbn_pipeline/google_books/9782203001190.json'),
 WindowsPath('c:/Users/yoann/Documents/mes projets/CollectionLens V2/data/raw/isbn_pipeline/google_books/9782351423554.json'),
 WindowsPath('c:/Users/yoann/Documents/mes projets/CollectionLens V2/data/raw/isbn_pipeline/google_books/9791039143431.json'),
 WindowsPath('c:/Users/yoann/Documents/mes projets/CollectionLens V2/data/raw/isbn_pipeline/nudger/9782203001190.json'),
 WindowsPath('c:/Users/yoann/Documents/mes projets/CollectionLens V2/data/raw/isbn_pipeline/nudger/9782351423554.json'),
 WindowsPath('c:/Users/